In [1]:
!pip install supar openai sentencepiece wandb nltk spacy huggingface_hub rouge-score bert-score numpy language_tool_python==2.9.2 scikit-learn scipy tqdm bitsandbytes transformers Pillow ftfy regex flake8 yapf isort yacs gdown tb-nightly future wilds tabulate torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.3/908.3 MB 1.8 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 109.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.6 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.0 MB/s eta 0:

In [5]:
#!/usr/bin/env python

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True' # Add this early

import collections # For deque
# ... other existing imports ...

# Global state for proactive rate limiting (REMOVED for local Bloomz model)

import time, gc, re, json, random, string, heapq, logging, argparse
import math
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize # Keep for urdu_sent_tokenize fallback if Stanza fails
from scipy.stats import entropy
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from bert_score import score as bert_score
import difflib
from nltk.metrics.distance import edit_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt
import unicodedata

# Suppress Warnings
import warnings
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from tqdm import tqdm

# Urdu-specific imports
try:
    import stanza
    stanza.download('ur', verbose=False)  # Download Urdu model
except:
    print("Stanza not available, using fallback methods for tokenization/sentencization")

try:
    import language_tool_python
    # Initialize LanguageTool for Urdu.
    # It's better to initialize it once globally.
    # This might download language files the first time.
    # Ensure Java is installed on your system for LanguageTool.
    urdu_lang_tool = language_tool_python.LanguageTool('ur-PK') # Or 'ur'
    print("LanguageTool for Urdu (ur-PK) initialized successfully.")
except ImportError:
    print("language-tool-python library not found. Please install it: pip install language-tool-python")
    print("LanguageTool-based grammar scoring will use fallback.")
    urdu_lang_tool = None
except Exception as e:
    print(f"Error initializing LanguageTool for Urdu: {e}.")
    print("Ensure Java is installed. LanguageTool-based grammar scoring will use fallback if init fails.")
    urdu_lang_tool = None

# Argument Parsing
parser = argparse.ArgumentParser(description='Multi-objective Genetic Search for Prompt Optimization - Urdu')
parser.add_argument('--mode', default="Instruction Only", help='Mode of instructions/prompts')
parser.add_argument('--num-shots', default=0, type=int, help='Number of examples in the prompt')
parser.add_argument('--batch-size', default=1, type=int, help='Batch size (influences data grouping, not direct model batching here)')
parser.add_argument('--task-idx', default=0, type=int, help='Index of the task')
parser.add_argument('--seed', default=3, type=int, help='Seed for sampling examples')
parser.add_argument('--train-seed', default=42, type=int, help='Seed for splitting and sampling edit operations')
parser.add_argument('--num-compose', default=1, type=int, help='Number of edits composed for one candidate')
parser.add_argument('--num-train', default=25, type=int, help='Number of examples in score set for GA evaluation')
parser.add_argument('--level', default="phrase", help='Level for edit operations')
parser.add_argument('--agnostic', action='store_true', default=False, help='Use template task-agnostic instruction')
parser.add_argument('--print-orig', action='store_true', default=True, help='Print original instruction and evaluate its performance')
parser.add_argument('--write-preds', action='store_true', default=True, help='Store predictions in a JSON file')
parser.add_argument('--meta-dir', default='/kaggle/working/logs/', help='Directory to store metadata')
parser.add_argument('--meta-name', default='urdu_bloomz_mutation_search.txt', help='Metadata filename')
parser.add_argument('--patience', default=5, type=int, help='Max patience')
parser.add_argument('--num-candidates', default=5, type=int, help='Number of candidates per iteration')
parser.add_argument('--num-iter', default=10, type=int, help='Maximum iterations')
parser.add_argument('--edits', nargs="+", default=['del', 'swap', 'sub', 'add'], help='Edit operations')
parser.add_argument('--tournament-selection', default=2, type=int, help='Number of tournament selections')
parser.add_argument('--population-size', default=10, type=int, help='Population size for GA')
parser.add_argument('--num-offspring', default=0, type=int, help='Number of offspring per iteration')
parser.add_argument('--mutation-prob', default=0.5, type=float, help='Mutation probability')
parser.add_argument('--data-dir', default='/kaggle/input/urdu-xlsum/', help='Dataset directory')
parser.add_argument('--project-name', default='Urdu_Bloomz_summarization_optimization', help='WandB project name')
parser.add_argument('--budget', default=2400, type=int, help='Model inference budget')
parser.add_argument('--bloomz-model-name', default="bigscience/bloomz-7b1", help='Name of the Bloomz model to use')

args, unknown = parser.parse_known_args()

os.makedirs(args.meta_dir, exist_ok=True)

for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    if os.path.exists(os.path.join(args.meta_dir, fname)):
        open(os.path.join(args.meta_dir, fname), 'w').close()
    else:
        with open(os.path.join(args.meta_dir, fname), 'w') as f:
            pass

try:
    urdu_nlp = stanza.Pipeline('ur', processors='tokenize,pos,lemma,depparse', verbose=False)
    print("Stanza Urdu pipeline initialized successfully")
except Exception as e:
    urdu_nlp = None
    print(f"Stanza not available or failed to initialize: {e}, using fallback tokenization")

global_progress_bar = tqdm(total=args.budget, desc="Model Inferences", leave=False, position=0, dynamic_ncols=True)

try:
    import wandb
    wandb.login(key='5dadd30326caa9de0ea95bb6f1c401782f469e83')
    wandb.init(project=args.project_name)
    wandb.config.update(args)
    tqdm.write("Wandb is setup\n")
except Exception as e:
    tqdm.write(f"Wandb initialization error: {e}\n")

torch.random.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

bloomz_model_name = args.bloomz_model_name
tokenizer = None
model = None

try:
    tqdm.write(f"Loading Bloomz tokenizer: {bloomz_model_name}")
    tokenizer = AutoTokenizer.from_pretrained(bloomz_model_name)
    tqdm.write(f"Loading Bloomz model: {bloomz_model_name} (this may take a while)...")
    model = AutoModelForCausalLM.from_pretrained(
        bloomz_model_name,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map="auto"
    )
    model.eval()
    tqdm.write(f"Bloomz model {bloomz_model_name} loaded successfully.")
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        tqdm.write("CUDA out of memory during Bloomz model initialization. Ensure you have enough GPU VRAM.")
        tqdm.write("Try a smaller model or run on a machine with more VRAM.")
        torch.cuda.empty_cache()
        gc.collect()
    raise e
except Exception as e:
    tqdm.write(f"Error loading Bloomz model: {e}")
    raise e

if torch.cuda.is_available() and model:
    tqdm.write("GPU Utilization after loading Bloomz:")
    for i in range(torch.cuda.device_count()):
        try:
            tqdm.write(f"GPU {i} ({torch.cuda.get_device_name(i)}): {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
                       f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved")
        except Exception as e:
            tqdm.write(f"Could not get memory info for GPU {i}: {e}")
    if hasattr(model, 'device'):
         tqdm.write(f"Bloomz model is primarily on device: {model.device}")
elif model:
    tqdm.write(f"CUDA not available or model not fully on GPU. Bloomz running on device: {model.device if hasattr(model, 'device') else 'CPU (assumed)'} (this may be slow).")
else:
    tqdm.write("Bloomz model not loaded. Cannot check GPU utilization.")

MAX_ARTICLE_CHARS_INPUT = 20000
evaluation_cache = {}

def is_urdu_text(text):
    if not isinstance(text, str): return False
    urdu_range = range(0x0600, 0x06FF)
    return any(ord(char) in urdu_range for char in text)

def enhanced_urdu_tokenize(text):
    if not isinstance(text, str) or not text.strip(): return []
    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            return [word.text for sentence in doc.sentences for word in sentence.words if word.text.strip()]
        except Exception: pass
    tokens = re.findall(r'[\u0600-\u06FF\u0750-\u077F]+|[^\s\u0600-\u06FF\u0750-\u077F]+', text)
    return [token.strip() for token in tokens if token.strip()]

def urdu_tokenize(text):
    return enhanced_urdu_tokenize(text)

def urdu_detokenize(tokens):
    if not tokens: return ""
    result = tokens[0]
    for token in tokens[1:]:
        if not re.match(r'^[۔؟!،؍]', token) and not unicodedata.category(token[0]).startswith('M'):
            result += " " + token
        else: result += token
    return result

def urdu_sent_tokenize(text):
    if not isinstance(text, str): return []
    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            return [sentence.text for sentence in doc.sentences]
        except Exception: pass
    sentences = re.split(r'([۔؟!])', text)
    result = []
    current_sentence = ""
    for part in sentences:
        current_sentence += part
        if part in '۔؟!':
            result.append(current_sentence.strip()); current_sentence = ""
    if current_sentence.strip(): result.append(current_sentence.strip())
    return [s for s in result if s]

def is_meaningful_phrase(phrase, is_single_word=False):
    if not phrase or not phrase.strip():
        return False
    phrase_cleaned = phrase.strip()
    if is_single_word:
        if not is_urdu_text(phrase_cleaned):
            return False
        if len(phrase_cleaned) < 1:
            return False
        single_stop_words = {
            'ہے', 'ہیں', 'ہوں', 'گا', 'گی', 'گے', 'تھا', 'تھی', 'تھے',
            'کا', 'کی', 'کے', 'کو', 'پر', 'میں', 'سے', 'تک', 'تک', 'یہ', 'وہ',
            'اس', 'ان', 'انہیں', 'انہوں', 'میرا', 'تیرا', 'ہمارا', 'آپکا',
            'اور', 'اگر', 'مگر', 'لیکن', 'بلکہ', 'تو', 'پھر', 'بھی', 'ہی',
            'ایک', 'دو', 'تین',
            'کر', 'ہو', 'جا'
        }
        if phrase_cleaned in single_stop_words:
            return False
    elif len(phrase_cleaned) < 3:
        return False
    tokens = enhanced_urdu_tokenize(phrase_cleaned)
    if not tokens:
        return False
    if is_single_word and len(tokens) != 1:
        return False
    if not is_single_word and len(tokens) < 2:
         return False
    if not is_single_word:
        content_tokens = [t for t in tokens if is_urdu_text(t) and not t.isnumeric()]
        if not content_tokens:
            return False
    multi_word_stop_phrases = {
        'اس کے', 'اس میں', 'یہ ہے', 'وہ ہے', 'کے لیے', 'کے ساتھ', 'کی وجہ سے',
        'دیا گیا', 'لیا گیا', 'کیا گیا', 'ہو رہا', 'جا رہا'
    }
    if not is_single_word and phrase_cleaned in multi_word_stop_phrases:
        return False
    if not is_single_word:
        non_urdu_char_count = sum(1 for char in phrase_cleaned if not ('\u0600' <= char <= '\u06FF' or char.isspace()))
        if non_urdu_char_count > len(phrase_cleaned) / 2:
            return False
    return True

def _fallback_phrase_extraction(sentence_text, phrases_list):
    tokens = enhanced_urdu_tokenize(sentence_text)
    if not tokens:
        return
    for token in tokens:
        if is_meaningful_phrase(token, is_single_word=True):
            phrases_list.append(token)
    if len(tokens) >= 2:
        for i in range(len(tokens) - 1):
            if is_urdu_text(tokens[i]) and is_urdu_text(tokens[i+1]):
                phrase_candidate = urdu_detokenize(tokens[i:i+2])
                if is_meaningful_phrase(phrase_candidate, is_single_word=False):
                    phrases_list.append(phrase_candidate)
    if len(tokens) >= 3:
        for i in range(len(tokens) - 2):
            if is_urdu_text(tokens[i]) and is_urdu_text(tokens[i+1]) and is_urdu_text(tokens[i+2]):
                phrase_candidate = urdu_detokenize(tokens[i:i+3])
                if is_meaningful_phrase(phrase_candidate, is_single_word=False):
                    phrases_list.append(phrase_candidate)

def get_enhanced_urdu_phrases(instruction):
    phrases = []
    if not isinstance(instruction, str) or not instruction.strip():
        return []
    sentences = urdu_sent_tokenize(instruction)
    content_upos_tags = ['NOUN', 'ADJ', 'VERB', 'ADV', 'PROPN', 'NUM']
    potential_single_word_tags = content_upos_tags + ['PRON', 'DET', 'PART']
    for sentence_text in sentences:
        if not sentence_text.strip():
            continue
        if urdu_nlp:
            try:
                doc = urdu_nlp(sentence_text)
                for sent_obj in doc.sentences:
                    current_phrase_tokens = []
                    for i, word in enumerate(sent_obj.words):
                        word_text = word.text.strip()
                        if not word_text:
                            continue
                        if word.upos in potential_single_word_tags:
                            if is_meaningful_phrase(word_text, is_single_word=True):
                                phrases.append(word_text)
                        is_core_content = word.upos in content_upos_tags
                        is_aux_or_neg_particle = word.upos == 'AUX' or \
                                                 (word.upos == 'PART' and word_text == 'نہیں')
                        if is_core_content:
                            current_phrase_tokens.append(word_text)
                        elif current_phrase_tokens and is_aux_or_neg_particle:
                            current_phrase_tokens.append(word_text)
                        else:
                            if len(current_phrase_tokens) >= 2:
                                phrase = urdu_detokenize(current_phrase_tokens)
                                if is_meaningful_phrase(phrase, is_single_word=False):
                                    phrases.append(phrase)
                            current_phrase_tokens = []
                            if is_core_content:
                                current_phrase_tokens.append(word_text)
                    if len(current_phrase_tokens) >= 2:
                        phrase = urdu_detokenize(current_phrase_tokens)
                        if is_meaningful_phrase(phrase, is_single_word=False):
                            phrases.append(phrase)
            except Exception as e_stanza:
                _fallback_phrase_extraction(sentence_text, phrases)
        else:
            _fallback_phrase_extraction(sentence_text, phrases)
    return filter_and_deduplicate_phrases(phrases)

def filter_and_deduplicate_phrases(phrases):
    if not phrases:
        return []
    meaningful_candidates_text = []
    seen_initial_texts = set()
    for p_text in phrases:
        stripped_p_text = p_text.strip()
        if not stripped_p_text or stripped_p_text in seen_initial_texts:
            continue
        p_tokens = enhanced_urdu_tokenize(stripped_p_text)
        is_single = len(p_tokens) == 1
        if is_meaningful_phrase(stripped_p_text, is_single_word=is_single):
            meaningful_candidates_text.append(stripped_p_text)
            seen_initial_texts.add(stripped_p_text)
    if not meaningful_candidates_text:
        return []
    meaningful_candidates_text.sort(key=lambda p: (len(enhanced_urdu_tokenize(p)), p), reverse=True)
    final_phrases_text = []
    accepted_phrase_tokens_list = []
    for phrase_text in meaningful_candidates_text:
        current_phrase_tokens_normalized = enhanced_urdu_tokenize(phrase_text.lower())
        if not current_phrase_tokens_normalized:
            continue
        is_exact_duplicate = False
        for accepted_tokens in accepted_phrase_tokens_list:
            if accepted_tokens == current_phrase_tokens_normalized:
                is_exact_duplicate = True
                break
        if is_exact_duplicate:
            continue
        is_sub_phrase = False
        if accepted_phrase_tokens_list:
            for accepted_tokens in accepted_phrase_tokens_list:
                if len(current_phrase_tokens_normalized) < len(accepted_tokens):
                    for i in range(len(accepted_tokens) - len(current_phrase_tokens_normalized) + 1):
                        if accepted_tokens[i:i+len(current_phrase_tokens_normalized)] == current_phrase_tokens_normalized:
                            is_sub_phrase = True
                            break
                    if is_sub_phrase:
                        break
        if not is_sub_phrase:
            final_phrases_text.append(phrase_text)
            accepted_phrase_tokens_list.append(current_phrase_tokens_normalized)
    return final_phrases_text

def get_urdu_phrases(instruction):
   return get_enhanced_urdu_phrases(instruction)

def get_urdu_phrase_lookup(base_candidate):
    phrases = get_urdu_phrases(base_candidate)
    phrase_lookup = {i: phrase for i, phrase in enumerate(phrases)}
    return phrase_lookup

def normalize_urdu_text(text):
    if not text: return ""
    text = str(text).strip()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[۔،؍؎؏ؘؙؚؐؑؒؓؔؕؖؗ؛؜]", "", text)
    return text.strip()

def calculate_urdu_rouge(reference, generated):
    ref_normalized = normalize_urdu_text(reference)
    gen_normalized = normalize_urdu_text(generated)
    ref_tokens = ref_normalized.split()
    gen_tokens = gen_normalized.split()

    if not ref_tokens or not gen_tokens:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    ref_unigrams = set(ref_tokens)
    gen_unigrams = set(gen_tokens)
    if len(gen_unigrams) == 0 or len(ref_unigrams) == 0:
        rouge1_precision, rouge1_recall, rouge1 = 0.0, 0.0, 0.0
    else:
        rouge1_overlap = len(ref_unigrams.intersection(gen_unigrams))
        rouge1_precision = rouge1_overlap / len(gen_unigrams)
        rouge1_recall = rouge1_overlap / len(ref_unigrams)
        rouge1 = (2 * rouge1_precision * rouge1_recall / (rouge1_precision + rouge1_recall)) if (rouge1_precision + rouge1_recall) > 0 else 0.0

    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:])) if len(ref_tokens) > 1 else set()
    gen_bigrams = set(zip(gen_tokens, gen_tokens[1:])) if len(gen_tokens) > 1 else set()
    if len(gen_bigrams) == 0 or len(ref_bigrams) == 0:
        rouge2_precision, rouge2_recall, rouge2 = 0.0, 0.0, 0.0
    else:
        rouge2_overlap = len(ref_bigrams.intersection(gen_bigrams))
        rouge2_precision = rouge2_overlap / len(gen_bigrams)
        rouge2_recall = rouge2_overlap / len(ref_bigrams)
        rouge2 = (2 * rouge2_precision * rouge2_recall / (rouge2_precision + rouge2_recall)) if (rouge2_precision + rouge2_recall) > 0 else 0.0

    def lcs_length(X, Y):
        m, n = len(X), len(Y)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if X[i - 1] == Y[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[m][n]

    lcs_len = lcs_length(ref_tokens, gen_tokens)
    if len(gen_tokens) == 0 or len(ref_tokens) == 0:
        rougeL_precision, rougeL_recall, rougeL = 0.0, 0.0, 0.0
    else:
        rougeL_precision = lcs_len / len(gen_tokens)
        rougeL_recall = lcs_len / len(ref_tokens)
        rougeL = (2 * rougeL_precision * rougeL_recall / (rougeL_precision + rougeL_recall)) if (rougeL_precision + rougeL_recall) > 0 else 0.0
    return {"rouge1": rouge1, "rouge2": rouge2, "rougeL": rougeL}

# Fallback heuristic grammar scoring function (if LanguageTool fails)
def urdu_grammar_fallback(text: str) -> int:
    """Basic heuristic fallback for Urdu grammar scoring."""
    score = 100
    if not is_urdu_text(text): # is_urdu_text should be defined in your script
        score -= 50
    
    # urdu_sent_tokenize and urdu_tokenize should be defined in your script
    sentences = urdu_sent_tokenize(text)
    if not sentences:
        score -= 30
    else:
        if len(urdu_tokenize(text)) < 3: # Check total tokens if sentences exist
            score -= 25
    
    # Check for sentence terminators
    if not any(ending in text for ending in ['۔', '؟', '!']):
        score -= 20
        
    for sentence in sentences:
        tokens = urdu_tokenize(sentence)
        if len(tokens) < 2 and len(sentences) > 1: # Very short sentence in multi-sentence text
            score -= 10
        elif len(tokens) > 40: # Very long sentence
            score -= 15
            
    return max(0, score)

def get_urdu_grammar_score(
    urdu_text: str,
    initial_score: int = 100,
    penalties_by_category: dict = None,
    default_penalty: float = 5.0
) -> int:
    """
    Checks Urdu text using LanguageTool and computes a grammar score from 0 to 100.
    Uses a fallback heuristic if LanguageTool is not available or fails during check.
    """
    global urdu_lang_tool # Access the globally initialized tool

    if not isinstance(urdu_text, str) or not urdu_text.strip() or len(urdu_text.strip()) < 4:
        # tqdm.write(f"Text too short or invalid for grammar scoring: '{urdu_text[:20]}...'") # Optional debug
        return 0

    if urdu_lang_tool is None:
        # tqdm.write("LanguageTool not initialized. Using fallback grammar scoring.") # Optional debug
        return urdu_grammar_fallback(urdu_text)

    current_score = float(initial_score)
    
    if penalties_by_category is None:
        penalties_by_category = {
            'TYPOS': 2.0,                # Often used for spelling by LT community
            'SPELLING': 3.0,
            'GRAMMAR': 7.0,
            'AGREEMENT': 8.0,            # e.g., verb-subject agreement
            'PUNCTUATION': 1.5,
            'STYLE': 4.0,                # Stylistic issues
            'TYPOGRAPHY': 1.0,
            'CONFUSED_WORDS': 6.0,
            'REDUNDANCY': 3.0,
            'UNKNOWN_WORD_CONSERVATIVE': 2.5, # Often indicates a potential spelling issue
            'MORFOLOGIK': 5.0,           # Generic morphological errors from some LT rules
            # Add more specific Urdu categories if you identify them from LanguageTool's output.
            # You can inspect `match.category` and `match.ruleId` from the `matches`
            # list to discover more categories for Urdu.
        }

    try:
        matches = urdu_lang_tool.check(urdu_text)
    except Exception as e:
        tqdm.write(f"Error during LanguageTool.check for text '{urdu_text[:30]}...': {e}. Using fallback.")
        return urdu_grammar_fallback(urdu_text)

    for match in matches:
        category = match.category.upper() # Normalize to uppercase for dict lookup
        penalty = penalties_by_category.get(category, default_penalty)
        current_score -= penalty
    
    final_score = max(0, current_score)
    final_score = min(initial_score, final_score) # Cannot exceed initial score
    
    return math.ceil(final_score)

def clean_generated_paraphrase(generated_text, original_phrase):
    if not generated_text:
        return original_phrase
    prefixes_to_remove = [
        "متبادل عبارت:", "متبادل:", "جواب:", "answer:",
        "paraphrase:", "alternative:", "یہ ہے:", "یہ:",
        "Paraphrased phrase:"
    ]
    suffixes_to_remove = ["۔", ".", "!", "؟", "?"]
    cleaned = generated_text.strip()
    for prefix in prefixes_to_remove:
        if cleaned.lower().startswith(prefix.lower()):
            cleaned = cleaned[len(prefix):].strip()
    cleaned = cleaned.strip('\'"\"\'')
    for suffix in suffixes_to_remove:
        if cleaned.endswith(suffix):
            cleaned = cleaned[:-len(suffix)].strip()
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned if cleaned else original_phrase

def is_valid_paraphrase(paraphrase, original_phrase, full_context):
    if not paraphrase or not paraphrase.strip():
        return False
    if paraphrase.strip().lower() == original_phrase.strip().lower():
        return False
    if not is_urdu_text(paraphrase):
        return False
    original_tokens = urdu_tokenize(original_phrase)
    paraphrase_tokens = urdu_tokenize(paraphrase)
    if len(paraphrase_tokens) > len(original_tokens) * 2.5 or len(paraphrase_tokens) < max(1, len(original_tokens) // 2):
        return False
    if original_phrase.lower() in paraphrase.lower() and paraphrase.lower() != original_phrase.lower():
        return False
    if len(original_tokens) <= 3 and any(punct in paraphrase for punct in ['۔', '؟', '!']):
        return False
    return True

# def get_bloomz_urdu_paraphrase(phrase_to_paraphrase, context_candidate=""):
#     if not isinstance(phrase_to_paraphrase, str) or not phrase_to_paraphrase.strip():
#         return phrase_to_paraphrase
#     if context_candidate:
#         prompt_for_paraphrase = (
#             f"سیاق و سباق: \"\"\"{context_candidate}\"\"\"\n"
#             f"اصل عبارت: \"\"\"{phrase_to_paraphrase}\"\"\"\n"
#             "سیاق و سباق کو مدنظر رکھتے ہوئے اصل عبارت کا ایک بہتر اور زیادہ موزوں متبادل فراہم کریں۔ صرف متبادل عبارت لکھیں۔\n"
#             "متبادل عبارت:"
#         )
#     else:
#         prompt_for_paraphrase = (
#             f"اردو عبارت \"\"\"{phrase_to_paraphrase}\"\"\" کا ایک بہتر اور زیادہ موزوں متبادل فراہم کریں۔\n"
#             "آپ کا جواب صرف متبادل عبارت پر مشتمل ہونا چاہیے۔ کوئی اضافی وضاحت یا تمہید شامل نہ کریں۔\n"
#             "متبادل عبارت:"
#         )
#     try:
#         paraphrase = generate_with_bloomz(
#             prompt_for_paraphrase,
#             max_new_tokens=70,
#             temperature=0.7,
#             do_sample=True,
#             operation_name="paraphrasing_via_bloomz"
#         )
#         return paraphrase.strip()
#     except Exception as e:
#         tqdm.write(f"Error during Bloomz paraphrasing for '{phrase_to_paraphrase}': {e}")
#         return phrase_to_paraphrase

def get_bloomz_urdu_paraphrase(phrase_to_paraphrase, context_candidate=""):
    if not isinstance(phrase_to_paraphrase, str) or not phrase_to_paraphrase.strip():
        return phrase_to_paraphrase

    # Option 1: Current Urdu Prompt (keep for comparison)
    # prompt_for_paraphrase_urdu = ... (your existing Urdu prompt logic)

    # Option 2: English Prompt for Instructions, Urdu for data
    if context_candidate:
        # Ensure context_candidate and phrase_to_paraphrase are clearly demarcated
        # and identified as Urdu.
        prompt_for_paraphrase_english_instruct = (
            f"Context (Urdu): \"\"\"{context_candidate}\"\"\"\n"
            f"Original phrase (Urdu): \"\"\"{phrase_to_paraphrase}\"\"\"\n"
            "Considering the context, provide a better and more suitable paraphrase for the original Urdu phrase. "
            "Your response should ONLY be the paraphrased Urdu phrase. Do not include any extra explanations or introductory text.\n"
            "Paraphrased Urdu phrase:"
        )
    else:
        prompt_for_paraphrase_english_instruct = (
            f"Original Urdu phrase: \"\"\"{phrase_to_paraphrase}\"\"\"\n"
            "Provide a better and more suitable paraphrase for this Urdu phrase. "
            "Your response should ONLY be the paraphrased Urdu phrase. Do not include any extra explanations or introductory text.\n"
            "Paraphrased Urdu phrase:"
        )

    # CHOOSE WHICH PROMPT TO USE FOR THE EXPERIMENT:
    # current_prompt_to_use = prompt_for_paraphrase_urdu
    current_prompt_to_use = prompt_for_paraphrase_english_instruct # For testing English instructions

    try:
        paraphrase = generate_with_bloomz(
            current_prompt_to_use, # Use the selected prompt
            max_new_tokens=70,     # Adjust if paraphrases are too short/long
            temperature=0.7,       # Good for creative but controlled output
            do_sample=True,
            operation_name="paraphrasing_via_bloomz"
        )
        # The clean_generated_paraphrase function might need to be aware
        # of potential English prefixes if the model accidentally adds them,
        # though the prompt strongly discourages it.
        return paraphrase.strip()
    except Exception as e:
        tqdm.write(f"Error during Bloomz paraphrasing for '{phrase_to_paraphrase}': {e}")
        return phrase_to_paraphrase

def substitute_urdu_phrase(candidate, phrase_to_replace):
    if not phrase_to_replace.strip():
        return candidate, phrase_to_replace, phrase_to_replace
    if phrase_to_replace not in candidate:
        return candidate, phrase_to_replace, phrase_to_replace

    original_phrase_for_return = phrase_to_replace
    try:
        generated_paraphrase = get_bloomz_urdu_paraphrase(phrase_to_replace, context_candidate=candidate)
        cleaned_paraphrase = clean_generated_paraphrase(generated_paraphrase, phrase_to_replace)

        if cleaned_paraphrase and is_valid_paraphrase(cleaned_paraphrase, phrase_to_replace, candidate):
            paraphrase_to_use = cleaned_paraphrase
            if phrase_to_replace in candidate:
                temp_candidate = candidate.replace(phrase_to_replace, paraphrase_to_use, 1)
            else:
                return candidate, original_phrase_for_return, original_phrase_for_return
            if temp_candidate == candidate:
                return candidate, original_phrase_for_return, original_phrase_for_return
            grammar_score = get_urdu_grammar_score(temp_candidate) # Now uses Bloomz with fallback
            if grammar_score >= 35:
                return temp_candidate, paraphrase_to_use, original_phrase_for_return
            else:
                return candidate, original_phrase_for_return, original_phrase_for_return
        else:
            return candidate, original_phrase_for_return, original_phrase_for_return
    except Exception as e:
        tqdm.write(f"Error during Bloomz paraphrasing substitution process for '{phrase_to_replace}': {e}")
        return candidate, original_phrase_for_return, original_phrase_for_return

def delete_urdu_phrase(candidate, phrase):
    if not phrase.strip(): return candidate
    patterns = [r'\s+' + re.escape(phrase) + r'\s+', r'\s+' + re.escape(phrase), re.escape(phrase) + r'\s+', re.escape(phrase)]
    new_candidate = candidate
    for i, pat_str in enumerate(patterns):
        replacement = ' ' if i == 0 else ''
        temp_candidate, num_subs = re.subn(pat_str, replacement, new_candidate, 1)
        if num_subs > 0:
            new_candidate = temp_candidate
            break
    return ' '.join(new_candidate.split())

def add_urdu_phrase(candidate, phrase_to_add, after_phrase):
    if not phrase_to_add.strip(): return candidate
    if not after_phrase.strip() or after_phrase not in candidate:
        return phrase_to_add + " " + candidate
    else:
        return candidate.replace(after_phrase, after_phrase + " " + phrase_to_add, 1)

def swap_urdu_phrases(candidate, phrase_1, phrase_2):
    if not phrase_1.strip() or not phrase_2.strip() or phrase_1 == phrase_2: return candidate
    placeholder1 = "___PLACEHOLDER_SWAP_1___" + str(random.randint(1000,9999))
    placeholder2 = "___PLACEHOLDER_SWAP_2___" + str(random.randint(1000,9999))
    idx1 = candidate.find(phrase_1)
    idx2 = candidate.find(phrase_2)
    if idx1 == -1 or idx2 == -1: return candidate
    temp_candidate = candidate
    if idx1 < idx2 :
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
    else:
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)
    final_candidate = temp_candidate.replace(placeholder1, phrase_2, 1)
    final_candidate = final_candidate.replace(placeholder2, phrase_1, 1)
    return final_candidate

def perform_urdu_edit(edit, base_text, phrase_list_full, deleted_phrases_history):
    mutated = base_text
    phrase_list = list(phrase_list_full)
    edit_description = f"Attempted {edit}: "
    if edit == 'del':
        if not phrase_list:
            edit_description += "No matching Urdu phrase for deletion."
            return base_text, deleted_phrases_history, edit_description
        chosen = random.choice(phrase_list)
        mutated = delete_urdu_phrase(base_text, chosen)
        if mutated != base_text:
            if chosen not in deleted_phrases_history: deleted_phrases_history.append(chosen)
            edit_description += f"Deleted phrase '{chosen}'."
        else: edit_description += f"Failed to delete phrase '{chosen}'."
    elif edit == 'swap':
        if len(phrase_list) < 2:
            edit_description += "Not enough matching Urdu phrases for swap."
            return base_text, deleted_phrases_history, edit_description
        p1, p2 = random.sample(phrase_list, 2)
        mutated = swap_urdu_phrases(base_text, p1, p2)
        if mutated != base_text: edit_description += f"Swapped '{p1}' with '{p2}'."
        else: edit_description += f"Failed to swap '{p1}' and '{p2}'."
    elif edit == 'sub':
        if not phrase_list:
            edit_description += "No matching Urdu phrase for substitution."
            return base_text, deleted_phrases_history, edit_description
        chosen_phrase_to_replace = random.choice(phrase_list)
        mutated, paraphrase_used, original_phrase = substitute_urdu_phrase(base_text, chosen_phrase_to_replace)
        if mutated != base_text and paraphrase_used != original_phrase:
            edit_description += f"Substituted '{original_phrase}' with '{paraphrase_used}'."
        elif paraphrase_used == original_phrase and mutated != base_text :
             edit_description += f"Replaced '{original_phrase}' with an identical phrase (formatting change)."
        elif paraphrase_used == original_phrase :
            edit_description += f"Substitution of '{original_phrase}' resulted in no change or was reverted."
        else:
            edit_description += f"Substitution of '{original_phrase}' with '{paraphrase_used}' was reverted."
    elif edit == 'add':
        if not deleted_phrases_history:
            edit_description += "No deleted Urdu phrases for addition."
            return base_text, deleted_phrases_history, edit_description
        phrase_to_add = random.choice(deleted_phrases_history)
        after = ""
        if phrase_list:
            after = random.choice(phrase_list)
            edit_description += f"Attempting to add '{phrase_to_add}' after '{after}'."
        else: edit_description += f"Attempting to prepend '{phrase_to_add}'."
        mutated = add_urdu_phrase(base_text, phrase_to_add, after)
        if mutated != base_text:
             if phrase_to_add in deleted_phrases_history: deleted_phrases_history.remove(phrase_to_add)
             edit_description = edit_description.replace("Attempting to add", "Successfully added")
        else: edit_description += " Addition resulted in no change."
    return mutated, deleted_phrases_history, edit_description

def safe_urdu_mutation(edit, base_text, phrase_list, deleted_phrases_history, grammar_threshold=30, meta_file_handle=None, mutation_step_info=""):
    mutated, new_del_history, edit_desc = perform_urdu_edit(edit, base_text, phrase_list, deleted_phrases_history)
    full_edit_log = f"    {mutation_step_info} - Op: {edit} - Detail: {edit_desc}"
    if mutated == base_text:
        log_entry = f"{full_edit_log} -> No change to candidate."
        tqdm.write(log_entry)
        if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
        return base_text, deleted_phrases_history
    gscore = get_urdu_grammar_score(mutated) # Now uses Bloomz with fallback
    if gscore >= grammar_threshold:
        log_entry = f"{full_edit_log} -> Accepted. New Grammar: {gscore}. Candidate: '{mutated[:70]}...'"
    else:
        log_entry = f"{full_edit_log} -> Rejected. Low Grammar: {gscore}. Reverting to: '{base_text[:70]}...'"
        mutated = base_text
        new_del_history = deleted_phrases_history
    tqdm.write(log_entry)
    if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
    return mutated, new_del_history

def encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(data_seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))
    random.seed(args.seed)
    test_indices = random.sample(instance_indices, min(number_of_instances, num_available_instances))
    generic_instruction_text = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        generic_instruction_text = "Definition: " + current_definition.strip()
    promptlist = []
    answerlist = []
    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_id = instances[i].get('id', f'test_idx_{i}')
        instance_input = raw_instance_input
        if len(raw_instance_input) > MAX_ARTICLE_CHARS_INPUT:
            sentences = urdu_sent_tokenize(raw_instance_input)
            truncated_input_parts = []
            current_len = 0
            for sent_text in sentences:
                if not sent_text.strip(): continue
                if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_INPUT:
                    truncated_input_parts.append(sent_text)
                    current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                else:
                    if not truncated_input_parts and sent_text:
                         truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_INPUT])
                    break
            instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_INPUT]
        system_message_content = "آپ ایک ماہر اردو متن کا خلاصہ نگار ہیں۔ آپ کو ایک عبارت اور اس کے لیے ہدایات دی جائیں گی۔ ہدایات پر عمل کرتے ہوئے ایک مختصر اور جامع خلاصہ فراہم کریں۔"
        user_content_parts = []
        if generic_instruction_text:
            user_content_parts.append(generic_instruction_text)
        user_content_parts.append("\nInput:\n" + instance_input)
        user_content_parts.append("\nخلاصہ:")
        user_content = "\n".join(user_content_parts)
        prompt_for_model_wrapper = [
            {"role": "system", "content": system_message_content},
            {"role": "user", "content": user_content}
        ]
        promptlist.append(prompt_for_model_wrapper)
        answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    return promptlist, answerlist, test_indices

def training_encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(args.seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))
    actual_num_test_instances = min(number_of_instances, num_available_instances)
    if actual_num_test_instances > num_available_instances:
        actual_num_test_instances = num_available_instances
    test_indices = random.sample(instance_indices, actual_num_test_instances)
    remaining_indices = [i for i in instance_indices if i not in test_indices]
    generic_instruction_text = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        generic_instruction_text = "Definition: " + current_definition.strip()
    system_message_content = "آپ ایک ماہر اردو متن کا خلاصہ نگار ہیں۔ آپ کو ایک عبارت اور اس کے لیے ہدایات دی جائیں گی۔ ہدایات پر عمل کرتے ہوئے ایک مختصر اور جامع خلاصہ فراہم کریں۔"
    test_promptlist = []
    test_answerlist = []
    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_input = raw_instance_input
        if len(raw_instance_input) > MAX_ARTICLE_CHARS_INPUT:
            sentences = urdu_sent_tokenize(raw_instance_input)
            truncated_input_parts = []
            current_len = 0
            for sent_text in sentences:
                if not sent_text.strip(): continue
                if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_INPUT:
                    truncated_input_parts.append(sent_text)
                    current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                else:
                    if not truncated_input_parts and sent_text:
                         truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_INPUT])
                    break
            instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_INPUT]
        user_content_parts = []
        if generic_instruction_text:
            user_content_parts.append(generic_instruction_text)
        user_content_parts.append("\nInput:\n" + instance_input)
        user_content_parts.append("\nخلاصہ:")
        user_content = "\n".join(user_content_parts)
        test_promptlist.append([
            {"role": "system", "content": system_message_content},
            {"role": "user", "content": user_content}
        ])
        test_answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    train_promptlist_full, train_answerlist_full, train_indexlist_full = [], [], []
    random.seed(args.train_seed)
    actual_num_train_samples = min(args.num_train, len(remaining_indices))
    if remaining_indices and actual_num_train_samples > 0 :
        train_sample_indices_from_remaining = random.sample(remaining_indices, actual_num_train_samples)
        for i in train_sample_indices_from_remaining:
            raw_instance_input = instances[i]['input']
            instance_input = raw_instance_input
            if len(raw_instance_input) > MAX_ARTICLE_CHARS_INPUT:
                sentences = urdu_sent_tokenize(raw_instance_input)
                truncated_input_parts = []
                current_len = 0
                for sent_text in sentences:
                    if not sent_text.strip(): continue
                    if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_INPUT:
                        truncated_input_parts.append(sent_text)
                        current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                    else:
                        if not truncated_input_parts and sent_text:
                             truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_INPUT])
                        break
                instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_INPUT]
            user_content_parts = []
            if generic_instruction_text:
                user_content_parts.append(generic_instruction_text)
            user_content_parts.append("\nInput:\n" + instance_input)
            user_content_parts.append("\nخلاصہ:")
            user_content = "\n".join(user_content_parts)
            train_promptlist_full.append([
                {"role": "system", "content": system_message_content},
                {"role": "user", "content": user_content}
            ])
            train_answerlist_full.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
            train_indexlist_full.append(i)
    dev_promptlist, dev_answerlist, dev_index_list = [], [], []
    return test_promptlist, test_answerlist, test_indices, train_promptlist_full, train_answerlist_full, train_indexlist_full, dev_promptlist, dev_answerlist, dev_index_list

def create_batches(instances_list, labels_list=[], batch_size=1):
    instance_batches, label_batches = [], []
    for i in range(0, len(instances_list), batch_size):
        instance_batches.append(instances_list[i:i+batch_size])
        if labels_list: label_batches.append(labels_list[i: i + batch_size])
    return (instance_batches, label_batches) if labels_list else instance_batches

def construct_urdu_instruction_prompt(mode, task_name, num_shots, num_test_instances, data_seed, null_word=None, args=args):
    if mode == "Instruction Only":
        prompt_list, answer_list, index_list = encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word, args=args)
    else: raise ValueError("Invalid mode entry, mode not recognized")
    return prompt_list, answer_list, index_list

def run(mode, batch_size, num_shots, chosen_task_name, num_samples, data_seed=0, override_prompts=False, function_to_call=None, split=None, modified={}, args=args):
    if not override_prompts:
        prompt_list, answer_list, index_list = construct_urdu_instruction_prompt(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, args=args)
    else:
        prompt_list, answer_list, index_list = function_to_call(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, null_word=None,
            split=split, modified=modified, args=args)
    prompt_batches, batch_test_labels = create_batches(prompt_list, answer_list, batch_size)
    predictions = []
    for batch_of_prompts, batch_of_labels in zip(prompt_batches, batch_test_labels):
        for individual_prompt_messages in batch_of_prompts:
            if generate_with_bloomz.count >= args.budget:
                tqdm.write("Budget exhausted in run function. Stopping generation.")
                predictions.extend([""] * (len(answer_list) - len(predictions)))
                break
            pred = get_bloomz_summary(individual_prompt_messages)
            predictions.append(pred)
        if generate_with_bloomz.count >= args.budget:
            break
    while len(predictions) < len(answer_list):
        predictions.append("")
    all_rouge_scores_dicts = []
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
            continue
        try: all_rouge_scores_dicts.append(calculate_urdu_rouge(ref, pred_text))
        except Exception as e:
            all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
    rouge1_scores_list = [s["rouge1"] for s in all_rouge_scores_dicts]
    rouge2_scores_list = [s["rouge2"] for s in all_rouge_scores_dicts]
    rougeL_scores_list = [s["rougeL"] for s in all_rouge_scores_dicts]
    valid_predictions = [p if p.strip() else "خالی" for p in predictions]
    valid_references = [r if r.strip() else "خالی" for r in answer_list]
    bert_f1_scores_list = [0.0] * len(answer_list)
    if valid_predictions and any(vp.strip() and vp != "خالی" for vp in valid_predictions):
        bert_device = "cuda" if torch.cuda.is_available() else "cpu"
        bert_model_type = "xlm-roberta-large"
        tqdm.write(f"Attempting BERTScore calculation on device: {bert_device} with model: {bert_model_type}")
        try:
            P_bert, R_bert, F1_bert = bert_score(
                valid_predictions, valid_references, lang="ur",
                model_type=bert_model_type, device=bert_device,
                verbose=False, batch_size=8 )
            bert_f1_scores_list = F1_bert.tolist()
            tqdm.write(f"BERTScore calculation successful on {bert_device}.")
        except Exception as e_gpu:
            tqdm.write(f"Error calculating BERT score on {bert_device} with model {bert_model_type}: {e_gpu}.")
            if bert_device == "cuda":
                tqdm.write("Falling back to CPU for BERTScore calculation.")
                try:
                    P_bert, R_bert, F1_bert = bert_score(
                        valid_predictions, valid_references, lang="ur",
                        model_type=bert_model_type, device="cpu",
                        verbose=False, batch_size=8 )
                    bert_f1_scores_list = F1_bert.tolist()
                    tqdm.write("BERTScore calculation successful on CPU (fallback).")
                except Exception as e_cpu:
                    tqdm.write(f"Error calculating BERT score on CPU (fallback) as well: {e_cpu}. Defaulting to 0.")
            else:
                 tqdm.write(f"BERTScore calculation failed on CPU. Defaulting to 0.")
        if not any(s > 0 for s in bert_f1_scores_list):
            tqdm.write("BERTScore resulted in all zeros, indicating a potential issue or all predictions were empty/invalid.")
    bleu_scores_list = []
    smoothie = SmoothingFunction().method4
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            bleu_scores_list.append(0.0)
            continue
        pred_tokens = urdu_tokenize(pred_text.lower())
        ref_tokens = urdu_tokenize(ref.lower())
        if not pred_tokens or not ref_tokens:
            bleu_scores_list.append(0.0)
            continue
        try:
            bleu_scores_list.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
        except Exception as e:
            bleu_scores_list.append(0.0)
    avg_rouge1 = np.mean(rouge1_scores_list) * 100 if rouge1_scores_list else 0.0
    avg_rouge2 = np.mean(rouge2_scores_list) * 100 if rouge2_scores_list else 0.0
    avg_rougeL = np.mean(rougeL_scores_list) * 100 if rougeL_scores_list else 0.0
    avg_bert = np.mean(bert_f1_scores_list) * 100 if bert_f1_scores_list else 0.0
    avg_bleu = np.mean(bleu_scores_list) * 100 if bleu_scores_list else 0.0
    return (predictions, avg_rouge1, avg_rouge2, avg_rougeL, avg_bert, avg_bleu,
            answer_list, index_list, rouge1_scores_list, rouge2_scores_list,
            rougeL_scores_list, bert_f1_scores_list, bleu_scores_list)

def counter(func):
    def wrapper(*args_wrapper, **kwargs_wrapper):
        wrapper.count += 1
        global_progress_bar.update(1)
        if wrapper.count >= args.budget:
            if global_progress_bar.n < global_progress_bar.total:
                global_progress_bar.n = global_progress_bar.total
                global_progress_bar.refresh()
            if not wrapper.budget_warning_shown:
                tqdm.write(f"Budget of {args.budget} model inferences exhausted (Inference #{wrapper.count}). Further inferences may fail or be skipped.")
                wrapper.budget_warning_shown = True
        return func(*args_wrapper, **kwargs_wrapper)
    wrapper.count = 0
    wrapper.budget_warning_shown = False
    return wrapper

@counter
def generate_with_bloomz(prompt_text, max_new_tokens=150, temperature=0.7, do_sample=True, operation_name="bloomz_generation"):
    global tokenizer, model, args
    if not tokenizer or not model:
        tqdm.write(f"Bloomz tokenizer or model not loaded. Cannot generate for {operation_name}.")
        return ""
    if generate_with_bloomz.count > args.budget: # Check budget before attempting, not after incrementing
        tqdm.write(f"Skipping {operation_name} due to exhausted budget (Count: {generate_with_bloomz.count}, Budget: {args.budget}).")
        return ""
    try:
        inputs = tokenizer.encode(prompt_text, return_tensors="pt").to(model.device)
        if do_sample and tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=do_sample,
                pad_token_id=tokenizer.pad_token_id
            )
        generated_text = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
        return generated_text.strip()
    except Exception as e:
        tqdm.write(f"Error during Bloomz model generation for {operation_name}: {type(e).__name__} - {e}")
        if "CUDA out of memory" in str(e):
            tqdm.write("CUDA OOM during generation. Try reducing input length, max_new_tokens, or batch size if applicable.")
            torch.cuda.empty_cache()
            gc.collect()
        return ""

def get_bloomz_summary(prompt_messages_or_string, max_tokens=None):
    prompt_text = ""
    if isinstance(prompt_messages_or_string, list) and all(isinstance(item, dict) for item in prompt_messages_or_string):
        system_content = ""
        user_content = ""
        for message in prompt_messages_or_string:
            if message.get("role") == "system":
                system_content += message.get("content", "") + "\n"
            elif message.get("role") == "user":
                user_content += message.get("content", "") + "\n"
        prompt_text = system_content.strip() + "\n" + user_content.strip()
        prompt_text = prompt_text.strip()
    elif isinstance(prompt_messages_or_string, str):
        prompt_text = prompt_messages_or_string
    else:
        tqdm.write(f"Warning: Unexpected prompt format in get_bloomz_summary: {type(prompt_messages_or_string)}")
        prompt_text = str(prompt_messages_or_string)
    effective_max_tokens = max_tokens if max_tokens is not None else 75
    return generate_with_bloomz(
        prompt_text,
        max_new_tokens=effective_max_tokens,
        temperature=0.5,
        do_sample=True,
        operation_name="summarization_via_bloomz"
    )

meta_path = os.path.join(args.meta_dir, args.meta_name)
meta_file_initial_open = open(meta_path, 'w+', encoding='utf-8')
batch_size = args.batch_size
num_shots = args.num_shots
mode = args.mode
seed = args.seed
train_seed = args.train_seed
model_name_for_log = args.bloomz_model_name
urdu_task_files = [f for f in os.listdir(args.data_dir) if f.endswith('.json')]
assert args.task_idx >= 0 and args.task_idx < len(urdu_task_files), "Invalid task index"
chosen_task_name = urdu_task_files[args.task_idx]
tqdm.write("Running Experiment for: " + chosen_task_name)
if 'wandb' in globals() and wandb.run:
    wandb.log({"Experiment": f"Running Experiment for: {chosen_task_name} with {model_name_for_log}", "Model": model_name_for_log})
nltk.download('punkt', quiet=True)
num_samples = 100
num_train_samples = args.num_train
np.random.seed(args.train_seed); torch.manual_seed(args.train_seed)
instruction = "دیے گئے متن کا ایک مناسب جملے میں خلاصہ بنائیں"
if args.agnostic:
    instruction = "آپ کو ایک متن دیا گیا ہے۔ اسے احتیاط سے پڑھیں اور سمجھیں، اور ایک مختصر خلاصہ فراہم کریں۔"
if 'wandb' in globals() and wandb.run:
    wandb.log({"Num Compose":args.num_compose,"Num Candidates":args.num_candidates,"Max Iterations":args.num_iter,
               "Tournament Selections":args.tournament_selection,"Edit Operations":args.edits,
               "Population Size":args.population_size,"Num Offspring":args.num_offspring,"Patience":args.patience,
               "Mutation Probability":args.mutation_prob})

def evaluate_objectives(candidate_prompt, split="train"):
    if candidate_prompt in evaluation_cache and split == evaluation_cache[candidate_prompt].get("split"):
        cached_data = evaluation_cache[candidate_prompt]
        return (cached_data["summarization_score"], cached_data["grammar_score"], cached_data["avg_rouge1"],
                cached_data["avg_rouge2"], cached_data["avg_rougeL"], cached_data["avg_bert"], cached_data["avg_bleu"])
    if generate_with_bloomz.count >= args.budget:
        tqdm.write("Budget exhausted in evaluate_objectives. Returning 0 scores.")
        return 0, 0, 0, 0, 0, 0, 0
    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, _, _, _, _, _, _, _) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name,
        num_samples=num_train_samples if split == "train" else num_samples,
        data_seed=args.seed if split == "test" else args.train_seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)
    summarization_score = (0.6 * avg_rL + 0.4 * avg_bert)
    grammar_score = get_urdu_grammar_score(candidate_prompt) # Now uses Bloomz with fallback
    with open(os.path.join(args.meta_dir, "rouge_scores.txt"), "a", encoding="utf-8") as f_rouge:
        f_rouge.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg R1: {avg_r1:.4f}, Avg R2: {avg_r2:.4f}, Avg RL: {avg_rL:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bert_scores.txt"), "a", encoding="utf-8") as f_bert:
        f_bert.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BERT: {avg_bert:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bleu_scores.txt"), "a", encoding="utf-8") as f_bleu:
        f_bleu.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BLEU: {avg_bleu:.4f}\n\n")
    evaluation_cache[candidate_prompt] = {
        "summarization_score": summarization_score, "grammar_score": grammar_score,
        "avg_rouge1": avg_r1, "avg_rouge2": avg_r2, "avg_rougeL": avg_rL,
        "avg_bert": avg_bert, "avg_bleu": avg_bleu, "split": split}
    return summarization_score, grammar_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu

def score_final(candidate_prompt, split="test", write=False):
    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, answer_list, indexlist,
     r1_scores_list, r2_scores_list, rL_scores_list, bert_scores_list, bleu_scores_list) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name, num_samples=num_samples, data_seed=args.seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)
    summarization_score = 0.6 * avg_rL + 0.4 * avg_bert
    if split == "test" and write:
        pname = args.meta_name.split(".")[0] + "_predictions.json"
        detailed_predictions_output = []
        for i in range(len(predictions)):
            detailed_predictions_output.append({
                "id": indexlist[i] if i < len(indexlist) else None,
                "reference_summary": answer_list[i] if i < len(answer_list) else None,
                "generated_summary": predictions[i] if i < len(predictions) else None,
                "rouge1": r1_scores_list[i] if i < len(r1_scores_list) else 0.0,
                "rouge2": r2_scores_list[i] if i < len(r2_scores_list) else 0.0,
                "rougeL": rL_scores_list[i] if i < len(rL_scores_list) else 0.0,
                "bert_score": bert_scores_list[i] if i < len(bert_scores_list) else 0.0,
                "bleu_score": bleu_scores_list[i] if i < len(bleu_scores_list) else 0.0,})
        pred_dump = {"final_prompt": candidate_prompt, "overall_avg_rouge1": avg_r1,
                     "overall_avg_rouge2": avg_r2, "overall_avg_rougeL": avg_rL,
                     "overall_avg_bert": avg_bert, "overall_avg_bleu": avg_bleu,
                     "predictions_detailed": detailed_predictions_output}
        ppath = os.path.join(args.meta_dir, pname)
        with open(ppath, "w+", encoding="utf-8") as pfile:
            json.dump(pred_dump, pfile, ensure_ascii=False, indent=2)
    return summarization_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu

def custom_urdu_instruction_prompt(mode=args.mode, task_name=None, num_shots=args.num_shots, num_test_instances=100, data_seed=None, null_word=None, split='train', modified={}, args=args):
    if task_name is None: task_name = chosen_task_name
    if mode == "Instruction Only":
        result = training_encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word,
            modified=modified, args=args)
    else: raise ValueError("Invalid mode for custom_urdu_instruction_prompt")
    if split == 'test': return result[0], result[1], result[2]
    elif split == 'train': return result[3], result[4], result[5]
    else: raise ValueError(f"Invalid split '{split}' for custom_urdu_instruction_prompt")

def tournament_selection(population, population_scores_tuples, num_tournaments):
    parent_indices = random.sample(range(len(population)), k=min(num_tournaments, len(population)))
    if not parent_indices:
        if population: return population[0], population_scores_tuples[0]
        return "", (0.0,0.0)
    best_parent_idx = -1
    best_parent_combined_score = -float('inf')
    for idx in parent_indices:
        s_score_val = population_scores_tuples[idx][0] if isinstance(population_scores_tuples[idx][0], (int, float)) else 0.0
        g_score_val = population_scores_tuples[idx][1] if isinstance(population_scores_tuples[idx][1], (int, float)) else 0.0
        combined_score = WEIGHT_SUMM * s_score_val + WEIGHT_GRAM * g_score_val
        if combined_score > best_parent_combined_score:
            best_parent_combined_score = combined_score
            best_parent_idx = idx
    if best_parent_idx == -1:
        best_parent_idx = parent_indices[0]
    return population[best_parent_idx], population_scores_tuples[best_parent_idx]

def urdu_crossover(parent_1_text, parent_2_text):
    try:
        phrases_1 = get_urdu_phrases(parent_1_text)
        phrases_2 = get_urdu_phrases(parent_2_text)
    except Exception: return parent_1_text, True
    if not phrases_1 or not phrases_2: return random.choice([parent_1_text, parent_2_text]), True
    p1_cross_point = random.randint(0, len(phrases_1))
    p2_cross_point = random.randint(0, len(phrases_2))
    offspring_phrases_part1 = phrases_1[:p1_cross_point] + phrases_2[p2_cross_point:]
    offspring_phrases_part2 = phrases_2[:p2_cross_point] + phrases_1[p1_cross_point:]
    chosen_offspring_phrases = random.choice([offspring_phrases_part1, offspring_phrases_part2])
    if not chosen_offspring_phrases: return random.choice([parent_1_text, parent_2_text]), True
    offspring_text = " ".join(chosen_offspring_phrases)
    offspring_text = ' '.join(urdu_tokenize(offspring_text))
    grammar_score = get_urdu_grammar_score(offspring_text) # Now uses Bloomz with fallback
    if is_urdu_text(offspring_text) and grammar_score >= 30: return offspring_text, False
    else: return random.choice([parent_1_text, parent_2_text]), True

def plot_pareto_front(population_data_tuples, fronts_indices, iteration, meta_dir_path, final=False):
    plt.figure(figsize=(10, 8))
    all_summ_scores = [data[1] for data in population_data_tuples]
    all_gram_scores = [data[2] for data in population_data_tuples]
    plt.scatter(all_summ_scores, all_gram_scores, c='lightgray', alpha=0.6, label='All Population Candidates', s=50)
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'olive', 'cyan']
    for front_idx, front_candidate_indices in enumerate(fronts_indices):
        if not front_candidate_indices: continue
        color = colors[front_idx % len(colors)]
        front_summ = [population_data_tuples[i][1] for i in front_candidate_indices]
        front_gram = [population_data_tuples[i][2] for i in front_candidate_indices]
        sorted_indices_within_front = np.argsort(front_summ)
        front_summ_sorted = np.array(front_summ)[sorted_indices_within_front]
        front_gram_sorted = np.array(front_gram)[sorted_indices_within_front]
        label = f'Pareto Front {front_idx + 1}' if front_idx > 0 else 'Pareto Optimal Front (F1)'
        plt.scatter(front_summ, front_gram, c=color, label=label, s=70, edgecolors='black')
        if len(front_summ_sorted) > 1 : plt.plot(front_summ_sorted, front_gram_sorted, c=color, linestyle='--', marker='o', markersize=5)
    plt.xlabel('Summarization Score (Higher is Better)'); plt.ylabel('Grammar Score (Higher is Better)')
    title_str = f'Pareto Optimal Fronts (Final)' if final else f'Pareto Optimal Fronts (Iteration {iteration+1})'
    plt.title(title_str, fontsize=16); plt.legend(loc='best'); plt.grid(True, linestyle=':', alpha=0.7); plt.tight_layout()
    plot_name = 'pareto_final.png' if final else f'pareto_iteration_{iteration+1}.png'
    plot_path = os.path.join(meta_dir_path, plot_name)
    plt.savefig(plot_path); plt.close()
    if 'wandb' in globals() and wandb.run:
        try: wandb.log({title_str: wandb.Image(plot_path)}, step=iteration if not final else args.num_iter +1)
        except Exception as e: tqdm.write(f"WandB logging error for Pareto plot: {e}")
    return plot_path

WEIGHT_SUMM = 0.7
WEIGHT_GRAM = 0.3
BEST_CANDIDATE_WEIGHT_SUMM = 0.8
BEST_CANDIDATE_WEIGHT_GRAM = 0.2

def non_dominated_sorting(population_data_list):
    n = len(population_data_list)
    if n == 0: return []
    domination_count = [0] * n
    dominated_solutions = [[] for _ in range(n)]
    fronts = [[]]
    for i in range(n):
        for j in range(i + 1, n):
            p_summ_raw, p_gram_raw = population_data_list[i][1], population_data_list[i][2]
            q_summ_raw, q_gram_raw = population_data_list[j][1], population_data_list[j][2]
            p_summ = p_summ_raw if isinstance(p_summ_raw, (int, float)) else -float('inf')
            p_gram = p_gram_raw if isinstance(p_gram_raw, (int, float)) else -float('inf')
            q_summ = q_summ_raw if isinstance(q_summ_raw, (int, float)) else -float('inf')
            q_gram = q_gram_raw if isinstance(q_gram_raw, (int, float)) else -float('inf')
            p_dominates_q = (p_summ >= q_summ and p_gram >= q_gram) and (p_summ > q_summ or p_gram > q_gram)
            q_dominates_p = (q_summ >= p_summ and q_gram >= p_gram) and (q_summ > p_summ or q_gram > q_gram)
            if p_dominates_q:
                dominated_solutions[i].append(j); domination_count[j] += 1
            elif q_dominates_p:
                dominated_solutions[j].append(i); domination_count[i] += 1
    for i in range(n):
        if domination_count[i] == 0: fronts[0].append(i)
    front_idx = 0
    while fronts[front_idx]:
        next_front_indices = []
        for i in fronts[front_idx]:
            for j in dominated_solutions[i]:
                domination_count[j] -= 1
                if domination_count[j] == 0: next_front_indices.append(j)
        front_idx += 1
        if next_front_indices: fronts.append(next_front_indices)
        else: break
    return fronts

def compute_crowding_distance(population_data_list, front_indices):
    if not front_indices: return {}
    num_in_front = len(front_indices)
    distances = {idx: 0.0 for idx in front_indices}
    num_objectives = 2
    for m in range(num_objectives):
        objective_idx_in_tuple = m + 1
        valid_indices_for_obj_m = [
            i for i in front_indices
            if isinstance(population_data_list[i][objective_idx_in_tuple], (int, float))
        ]
        if not valid_indices_for_obj_m: continue
        sorted_front_by_obj_m = sorted(valid_indices_for_obj_m, key=lambda i: population_data_list[i][objective_idx_in_tuple])
        if not sorted_front_by_obj_m: continue
        distances[sorted_front_by_obj_m[0]] = float('inf')
        if len(sorted_front_by_obj_m) > 1: distances[sorted_front_by_obj_m[-1]] = float('inf')
        range_obj_m = 0
        if len(sorted_front_by_obj_m) > 1:
            obj_m_min = population_data_list[sorted_front_by_obj_m[0]][objective_idx_in_tuple]
            obj_m_max = population_data_list[sorted_front_by_obj_m[-1]][objective_idx_in_tuple]
            range_obj_m = obj_m_max - obj_m_min
        if len(sorted_front_by_obj_m) > 2 and range_obj_m > 1e-6:
            for k in range(1, len(sorted_front_by_obj_m) - 1):
                idx_k = sorted_front_by_obj_m[k]
                idx_k_plus_1 = sorted_front_by_obj_m[k+1]
                idx_k_minus_1 = sorted_front_by_obj_m[k-1]
                distances[idx_k] += (population_data_list[idx_k_plus_1][objective_idx_in_tuple] -
                                     population_data_list[idx_k_minus_1][objective_idx_in_tuple]) / range_obj_m
    return distances

def plot_best_candidate_scores(meta_dir_path, r1_scores, r2_scores, rL_scores, b_scores, bl_scores, summ_scores, gram_scores, comb_scores):
    iterations = list(range(len(rL_scores)))
    score_types = {"ROUGE-1": r1_scores, "ROUGE-2": r2_scores, "ROUGE-L": rL_scores,
                   "BERT": b_scores, "BLEU": bl_scores, "Summarization": summ_scores,
                   "Grammar": gram_scores, "Combined (0.8S+0.2G)": comb_scores}
    for score_name, scores_data in score_types.items():
        plt.figure(figsize=(10, 6))
        plt.plot(iterations, scores_data, marker='o', linestyle='-', markersize=5, label=f'Best {score_name}')
        plt.xlabel('Iteration Number (0 = Initial)'); plt.ylabel(f'{score_name} Score')
        plt.title(f'Best Candidate {score_name} Score vs. Iteration', fontsize=14)
        plt.grid(True, linestyle=':', alpha=0.7); plt.xticks(iterations); plt.legend(); plt.tight_layout()
        plot_path = os.path.join(meta_dir_path, f'history_best_{score_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace(".","").replace("+","")}_scores.png')
        plt.savefig(plot_path); plt.close()
        if 'wandb' in globals() and wandb.run:
            try: wandb.log({f"History Best {score_name} Scores Plot": wandb.Image(plot_path)}, step=iterations[-1] if iterations else 0)
            except Exception as e: tqdm.write(f"WandB logging error for score history plot {score_name}: {e}")

def adaptive_mutation_probability(iteration, max_iterations, base_prob=0.5):
    decay_factor = 1 - (iteration / max_iterations) * 0.3
    return base_prob * decay_factor

W_candidates = [instruction] * args.population_size
W_deletesets = [[] for _ in range(args.population_size)]

with open(meta_path, 'a', encoding='utf-8') as meta_append_initial:
    meta_append_initial.write(f"Initial Urdu Candidate:\t {instruction}\n")
    tqdm.write(f"Initial Urdu Candidate: {instruction}")
    torch.cuda.empty_cache(); gc.collect()
    s_score, g_score, r1_score, r2_score, rL_score, b_score, bl_score = evaluate_objectives(instruction, split='train')
    W_objectives = [(s_score, g_score)] * args.population_size
    meta_append_initial.write(
        "Initial Objectives (Summarization, Grammar, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU):\t "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})\n")
    tqdm.write(
        "Initial Objectives (Summarization, Grammar, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU): "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})")

if 'wandb' in globals() and wandb.run:
    wandb.log({"Initial Urdu Candidate": instruction, "Initial Summarization Score": s_score, "Initial Grammar Score": g_score,
               "Initial ROUGE-1 Score": r1_score, "Initial ROUGE-2 Score": r2_score, "Initial ROUGE-L Score": rL_score,
               "Initial BERT Score": b_score, "Initial BLEU Score": bl_score, "Iteration": 0})

history_best_rouge1 = [r1_score]; history_best_rouge2 = [r2_score]; history_best_rougeL = [rL_score]
history_best_bert = [b_score]; history_best_bleu = [bl_score]
history_best_summarization = [s_score]; history_best_grammar = [g_score]
history_best_combined = [BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score]
overall_best_candidate_text = instruction
overall_best_candidate_objectives = (s_score, g_score)
overall_best_combined_score = BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score
patience_counter = 0; start_time = time.time(); iter_idx = 0

for iter_idx in range(args.num_iter):
    if generate_with_bloomz.count >= args.budget:
        tqdm.write("Budget exhausted before starting iteration. Stopping.")
        break
    with open(meta_path, 'a', encoding='utf-8') as current_iter_meta_file:
        current_iter_meta_file.write(f"\n----- Iteration: {iter_idx + 1} -----\n")
        tqdm.write(f"\n----- Iteration: {iter_idx + 1} -----")
        current_iter_meta_file.write("Population at START of Iteration:\n"); tqdm.write("Population at START of Iteration:")
        population_log_start = []
        for i_pop, cand_text in enumerate(W_candidates):
            current_s, current_g = W_objectives[i_pop] if isinstance(W_objectives[i_pop], tuple) and len(W_objectives[i_pop]) == 2 else (0.0, 0.0)
            pop_entry = {"prompt": cand_text, "summ_score": current_s, "gram_score": current_g}
            population_log_start.append(pop_entry)
            log_line = f"  Cand {i_pop+1}: Summ={current_s:.2f}, Gram={current_g:.2f} - '{cand_text[:70]}...'\n"
            current_iter_meta_file.write(log_line); tqdm.write(log_line.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population Start (Iter {iter_idx+1})": population_log_start}, step=iter_idx+1)
        offspring_candidates, offspring_deletesets = [], []
        if args.num_offspring > 0:
            for j_offspring in range(args.num_offspring // 2):
                if generate_with_bloomz.count >= args.budget: break
                parent1_text, _ = tournament_selection(W_candidates, W_objectives, args.tournament_selection)
                parent2_text, _ = tournament_selection(W_candidates, W_objectives, args.tournament_selection)
                if parent1_text == parent2_text and len(W_candidates) > 1:
                    temp_population_for_tournament = [c for c in W_candidates if c != parent1_text]
                    if temp_population_for_tournament:
                        temp_objectives_indices = [i for i, c in enumerate(W_candidates) if c != parent1_text]
                        temp_objectives_for_tournament = [W_objectives[idx] for idx in temp_objectives_indices]
                        if temp_objectives_for_tournament:
                           parent2_text, _ = tournament_selection(temp_population_for_tournament, temp_objectives_for_tournament, args.tournament_selection)
                current_iter_meta_file.write(f"  Parent 1 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent1_text}\n")
                current_iter_meta_file.write(f"  Parent 2 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent2_text}\n")
                child1_text, err1 = urdu_crossover(parent1_text, parent2_text)
                child2_text, err2 = urdu_crossover(parent2_text, parent1_text)
                if not err1 and child1_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child1_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent1_text)]) if parent1_text in W_candidates else [])
                if not err2 and child2_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child2_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent2_text)]) if parent2_text in W_candidates else [])
        mutated_candidates_texts, mutated_deletesets = [], []
        candidates_for_mutation = list(W_candidates)
        for i_mut in range(len(candidates_for_mutation)):
            if generate_with_bloomz.count >= args.budget: break
            current_mutation_prob = args.mutation_prob
            if random.random() < current_mutation_prob:
                base_candidate_text = candidates_for_mutation[i_mut]
                try:
                    original_idx_for_mutation = W_candidates.index(base_candidate_text)
                    base_deleteset = list(W_deletesets[original_idx_for_mutation])
                except ValueError:
                    base_deleteset = []
                current_iter_meta_file.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text}\n")
                tqdm.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text[:70]}...")
                mutated_text_current = base_candidate_text
                current_deleteset_for_mutation = list(base_deleteset)
                try:
                    phrase_list_for_mutation = get_urdu_phrases(base_candidate_text)
                    current_iter_meta_file.write(f"    Extracted Phrases for Mutation: {phrase_list_for_mutation}\n")
                    tqdm.write(f"    Extracted Phrases: {phrase_list_for_mutation}")
                except Exception as e:
                    current_iter_meta_file.write(f"    Error getting phrases for mutation: {e}\n"); tqdm.write(f"    Error getting phrases: {e}")
                    continue
                for edit_idx_compose in range(args.num_compose):
                    if not args.edits: continue
                    available_edits = list(args.edits)
                    if not phrase_list_for_mutation or len(phrase_list_for_mutation) < 2:
                        if 'swap' in available_edits: available_edits.remove('swap')
                    if not phrase_list_for_mutation:
                        if 'del' in available_edits: available_edits.remove('del')
                        if 'sub' in available_edits: available_edits.remove('sub')
                    if not current_deleteset_for_mutation and 'add' in available_edits: available_edits.remove('add')
                    if not available_edits:
                        current_iter_meta_file.write(f"    ComposeStep {edit_idx_compose+1}: No available edit operations.\n")
                        tqdm.write(f"    ComposeStep {edit_idx_compose+1}: No available edit operations.")
                        break
                    chosen_edit_op = random.choice(available_edits)
                    mutated_text_current, current_deleteset_for_mutation = safe_urdu_mutation(
                        chosen_edit_op, mutated_text_current, list(phrase_list_for_mutation),
                        current_deleteset_for_mutation, meta_file_handle=current_iter_meta_file,
                        mutation_step_info=f"Cand {i_mut+1} ComposeStep {edit_idx_compose+1}")
                    if mutated_text_current != base_candidate_text and edit_idx_compose < args.num_compose - 1 :
                        try: phrase_list_for_mutation = get_urdu_phrases(mutated_text_current)
                        except Exception: phrase_list_for_mutation = []
                if mutated_text_current != base_candidate_text and mutated_text_current not in mutated_candidates_texts + W_candidates + offspring_candidates:
                    mutated_candidates_texts.append(mutated_text_current)
                    mutated_deletesets.append(current_deleteset_for_mutation)
        combined_candidates_texts_pool = W_candidates + offspring_candidates + mutated_candidates_texts
        deletesets_pool_map = {}
        for i, txt in enumerate(W_candidates): deletesets_pool_map.setdefault(txt, W_deletesets[i])
        for i, txt in enumerate(offspring_candidates): deletesets_pool_map.setdefault(txt, offspring_deletesets[i])
        for i, txt in enumerate(mutated_candidates_texts): deletesets_pool_map.setdefault(txt, mutated_deletesets[i])
        unique_combined_texts_list = []
        seen_texts_for_eval = set()
        for txt_comb in combined_candidates_texts_pool:
            if txt_comb not in seen_texts_for_eval:
                unique_combined_texts_list.append(txt_comb)
                seen_texts_for_eval.add(txt_comb)
        final_combined_deletesets_for_eval = [deletesets_pool_map.get(text, []) for text in unique_combined_texts_list]
        tqdm.write(f"Evaluating {len(unique_combined_texts_list)} unique candidates in combined pool for Iter {iter_idx + 1}")
        evaluated_objectives_list = []
        all_candidate_scores_for_iter_log = []
        for i_eval, cand_text_eval in enumerate(unique_combined_texts_list):
            if generate_with_bloomz.count >= args.budget:
                num_remaining_to_eval = len(unique_combined_texts_list) - len(evaluated_objectives_list)
                evaluated_objectives_list.extend([(-1.0, -1.0)] * num_remaining_to_eval)
                all_candidate_scores_for_iter_log.extend([(0,0,0,0,0,-1.0,-1.0)] * num_remaining_to_eval)
                break
            s, g, r1, r2, rL, bert, bleu = evaluate_objectives(cand_text_eval, split='train')
            evaluated_objectives_list.append((s, g))
            all_candidate_scores_for_iter_log.append((r1, r2, rL, bert, bleu, s, g))
        current_population_data_for_sorting = []
        for i_data in range(len(unique_combined_texts_list)):
            summ_s, gram_s = evaluated_objectives_list[i_data]
            deleteset_val = final_combined_deletesets_for_eval[i_data]
            current_population_data_for_sorting.append(
                (unique_combined_texts_list[i_data], summ_s, gram_s, deleteset_val)
            )
        fronts = non_dominated_sorting(current_population_data_for_sorting)
        if fronts and fronts[0] and callable(globals().get('plot_pareto_front')):
            plot_pareto_front(current_population_data_for_sorting, fronts, iter_idx, args.meta_dir)
        next_gen_indices = []
        remaining_population_slots = args.population_size
        for front_indices_in_data in fronts:
            if not front_indices_in_data: continue
            if remaining_population_slots == 0: break
            if len(front_indices_in_data) <= remaining_population_slots:
                next_gen_indices.extend(front_indices_in_data)
                remaining_population_slots -= len(front_indices_in_data)
            else:
                if callable(globals().get('compute_crowding_distance')):
                    distances = compute_crowding_distance(current_population_data_for_sorting, front_indices_in_data)
                    sorted_front_by_crowding = sorted(front_indices_in_data, key=lambda i_crowd: distances.get(i_crowd, 0.0), reverse=True)
                    next_gen_indices.extend(sorted_front_by_crowding[:remaining_population_slots])
                else:
                    tqdm.write("Warning: compute_crowding_distance not found. Using simple truncation.")
                    next_gen_indices.extend(front_indices_in_data[:remaining_population_slots])
                remaining_population_slots = 0
        temp_W_candidates = [current_population_data_for_sorting[i][0] for i in next_gen_indices]
        temp_W_objectives = [(current_population_data_for_sorting[i][1], current_population_data_for_sorting[i][2]) for i in next_gen_indices]
        temp_W_deletesets = [current_population_data_for_sorting[i][3] for i in next_gen_indices]
        final_W_candidates = []
        final_W_objectives = []
        final_W_deletesets = []
        num_actually_selected = len(temp_W_candidates)
        if num_actually_selected >= args.population_size:
            final_W_candidates = temp_W_candidates[:args.population_size]
            final_W_objectives = temp_W_objectives[:args.population_size]
            final_W_deletesets = temp_W_deletesets[:args.population_size]
        else:
            final_W_candidates.extend(temp_W_candidates)
            final_W_objectives.extend(temp_W_objectives)
            final_W_deletesets.extend(temp_W_deletesets)
            padding_needed = args.population_size - num_actually_selected
            if num_actually_selected > 0:
                padding_source_with_scores = []
                for i_pad_src in range(num_actually_selected):
                    s_pad, g_pad = temp_W_objectives[i_pad_src]
                    s_pad_num = s_pad if isinstance(s_pad, (int, float)) else 0.0
                    g_pad_num = g_pad if isinstance(g_pad, (int, float)) else 0.0
                    combined_score_pad = WEIGHT_SUMM * s_pad_num + WEIGHT_GRAM * g_pad_num
                    padding_source_with_scores.append(
                        (temp_W_candidates[i_pad_src], temp_W_objectives[i_pad_src], temp_W_deletesets[i_pad_src], combined_score_pad)
                    )
                padding_source_with_scores.sort(key=lambda x_pad: x_pad[3], reverse=True)
                for i_pad in range(padding_needed):
                    source_idx_pad = i_pad % len(padding_source_with_scores)
                    final_W_candidates.append(padding_source_with_scores[source_idx_pad][0])
                    final_W_objectives.append(padding_source_with_scores[source_idx_pad][1])
                    final_W_deletesets.append(padding_source_with_scores[source_idx_pad][2])
            else:
                tqdm.write(f"Warning: No candidates selected by Pareto. Padding with initial instruction.")
                initial_obj_tuple_for_pad = (s_score if 's_score' in locals() else 0.0, g_score if 'g_score' in locals() else 0.0)
                for _ in range(padding_needed):
                    final_W_candidates.append(instruction)
                    final_W_objectives.append(initial_obj_tuple_for_pad)
                    final_W_deletesets.append([])
        W_candidates = final_W_candidates
        W_objectives = final_W_objectives
        W_deletesets = final_W_deletesets
        current_iter_meta_file.write("Population at END of Iteration (Selected for Next Gen):\n"); tqdm.write("Population at END of Iteration (Selected for Next Gen):")
        population_log_end = []
        for i_pop_end, cand_text_end in enumerate(W_candidates):
            obj_s_end, obj_g_end = W_objectives[i_pop_end] if isinstance(W_objectives[i_pop_end], tuple) and len(W_objectives[i_pop_end]) == 2 else (0.0, 0.0)
            pop_entry_end = {"prompt": cand_text_end, "summ_score": obj_s_end, "gram_score": obj_g_end}
            population_log_end.append(pop_entry_end)
            log_line_end = f"  Cand {i_pop_end+1}: Summ={obj_s_end:.2f}, Gram={obj_g_end:.2f} - '{cand_text_end[:70]}...'\n"
            current_iter_meta_file.write(log_line_end); tqdm.write(log_line_end.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population End (Iter {iter_idx+1})": population_log_end}, step=iter_idx+1)
        iter_best_candidate_text, iter_best_objectives, iter_best_scores_tuple = "N/A", (-1.0,-1.0), (0,0,0,0,0,-1.0,-1.0)
        if fronts and fronts[0]:
            pareto_front_indices_for_log = fronts[0]
            best_idx_in_pareto_for_log = -1
            current_max_weighted_score_in_pareto_for_log = -float('inf')
            for idx_in_data_for_log in pareto_front_indices_for_log:
                s_val_log, g_val_log = current_population_data_for_sorting[idx_in_data_for_log][1], current_population_data_for_sorting[idx_in_data_for_log][2]
                s_val_log_num = s_val_log if isinstance(s_val_log, (int, float)) else 0.0
                g_val_log_num = g_val_log if isinstance(g_val_log, (int, float)) else 0.0
                weighted_score_log = BEST_CANDIDATE_WEIGHT_SUMM * s_val_log_num + BEST_CANDIDATE_WEIGHT_GRAM * g_val_log_num
                if weighted_score_log > current_max_weighted_score_in_pareto_for_log:
                    current_max_weighted_score_in_pareto_for_log = weighted_score_log
                    best_idx_in_pareto_for_log = idx_in_data_for_log
            if best_idx_in_pareto_for_log != -1:
                iter_best_candidate_text = current_population_data_for_sorting[best_idx_in_pareto_for_log][0]
                iter_best_objectives = (current_population_data_for_sorting[best_idx_in_pareto_for_log][1], current_population_data_for_sorting[best_idx_in_pareto_for_log][2])
                if best_idx_in_pareto_for_log < len(all_candidate_scores_for_iter_log):
                     iter_best_scores_tuple = all_candidate_scores_for_iter_log[best_idx_in_pareto_for_log]
                else:
                     iter_best_scores_tuple = (0,0,0,0,0, iter_best_objectives[0], iter_best_objectives[1])
        history_best_rouge1.append(iter_best_scores_tuple[0]); history_best_rouge2.append(iter_best_scores_tuple[1])
        history_best_rougeL.append(iter_best_scores_tuple[2]); history_best_bert.append(iter_best_scores_tuple[3])
        history_best_bleu.append(iter_best_scores_tuple[4]); history_best_summarization.append(iter_best_scores_tuple[5])
        history_best_grammar.append(iter_best_scores_tuple[6])
        current_iter_combined_score_for_history = BEST_CANDIDATE_WEIGHT_SUMM * iter_best_scores_tuple[5] + BEST_CANDIDATE_WEIGHT_GRAM * iter_best_scores_tuple[6]
        history_best_combined.append(current_iter_combined_score_for_history)
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Logged Best Candidate (0.8S+0.2G from F1): {iter_best_candidate_text}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Objectives (Summ, Gram): {iter_best_objectives}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Scores (R1,R2,RL,B,BL,S,G): {iter_best_scores_tuple}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Combined Score (0.8S+0.2G): {current_iter_combined_score_for_history:.2f}\n")
        tqdm.write(f"Iteration {iter_idx+1} - Logged Best Candidate (0.8S+0.2G from F1): {iter_best_candidate_text[:50]}... Objectives: {iter_best_objectives}, Combined (0.8/0.2): {current_iter_combined_score_for_history:.2f}")
        if 'wandb' in globals() and wandb.run:
            wandb.log({"Iteration Logged Best Candidate Text": iter_best_candidate_text,
                       "Iteration Logged Best Summarization Score": iter_best_scores_tuple[5],
                       "Iteration Logged Best Grammar Score": iter_best_scores_tuple[6],
                       "Iteration Logged Best ROUGE-1": iter_best_scores_tuple[0],
                       "Iteration Logged Best ROUGE-2": iter_best_scores_tuple[1],
                       "Iteration Logged Best ROUGE-L": iter_best_scores_tuple[2],
                       "Iteration Logged Best BERT": iter_best_scores_tuple[3],
                       "Iteration Logged Best BLEU": iter_best_scores_tuple[4],
                       "Iteration Logged Best Combined Score (0.8S+0.2G)": current_iter_combined_score_for_history,
                       "Model Inferences": generate_with_bloomz.count, "Iteration": iter_idx + 1})
        if current_iter_combined_score_for_history > overall_best_combined_score:
            overall_best_combined_score = current_iter_combined_score_for_history
            overall_best_candidate_text = iter_best_candidate_text
            overall_best_candidate_objectives = iter_best_objectives
            patience_counter = 0
            current_iter_meta_file.write(f"New Overall Best Candidate Found (based on 0.8S+0.2G)!\n")
            tqdm.write(f"New Overall Best Candidate Found (based on 0.8S+0.2G)! Score: {overall_best_combined_score:.2f}")
        else: patience_counter += 1
        if patience_counter >= args.patience:
            tqdm.write(f"Patience ({args.patience}) exhausted. Stopping early.")
            current_iter_meta_file.write("Patience exhausted. Stopping early.\n")
            break
    torch.cuda.empty_cache(); gc.collect()

with open(meta_path, 'a', encoding='utf-8') as final_meta_file:
    if iter_idx < args.num_iter -1 and generate_with_bloomz.count < args.budget and patience_counter < args.patience:
        pass
    elif 'current_population_data_for_sorting' in locals() and 'fronts' in locals() and fronts and fronts[0]:
         plot_pareto_front(current_population_data_for_sorting, fronts, iter_idx, args.meta_dir, final=True)
    else: tqdm.write("Could not plot final Pareto front (no data or fronts from last iteration).")
    final_meta_file.write(f"\nSearch Finished.\nModel Inferences for search: {generate_with_bloomz.count}\n")
    tqdm.write(f"Model Inferences for search: {generate_with_bloomz.count}")
    if 'wandb' in globals() and wandb.run: wandb.log({"Total Model Inferences (Search)": generate_with_bloomz.count})
    plot_best_candidate_scores(args.meta_dir, history_best_rouge1, history_best_rouge2, history_best_rougeL,
                               history_best_bert, history_best_bleu, history_best_summarization,
                               history_best_grammar, history_best_combined)
    tqdm.write("\nTesting the overall best candidate found...")
    final_meta_file.write("\nTesting the overall best candidate found...\n")
    final_s_score, final_r1_score, final_r2_score, final_rL_score, final_b_score, final_bl_score = score_final(
        overall_best_candidate_text, 'test', write=args.write_preds)
    tqdm.write(f"Final Overall Best Candidate (based on 0.8S+0.2G): {overall_best_candidate_text}")
    tqdm.write(f"Final Test Scores (Summarization, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU): ({final_s_score:.2f}, {final_r1_score:.2f}, {final_r2_score:.2f}, {final_rL_score:.2f}, {final_b_score:.2f}, {final_bl_score:.2f})")
    final_meta_file.write(f"Final Overall Best Candidate (based on 0.8S+0.2G): {overall_best_candidate_text}\n")
    final_meta_file.write(f"Final Test Summarization Score: {final_s_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-1 Score: {final_r1_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-2 Score: {final_r2_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-L Score: {final_rL_score:.2f}\n")
    final_meta_file.write(f"Final Test BERT Score: {final_b_score:.2f}\n")
    final_meta_file.write(f"Final Test BLEU Score: {final_bl_score:.2f}\n")
    if 'wandb' in globals() and wandb.run:
        wandb.log({"Final Overall Best Candidate Text": overall_best_candidate_text, "Final Test Summarization Score": final_s_score,
                   "Final Test ROUGE-1 Score": final_r1_score, "Final Test ROUGE-2 Score": final_r2_score,
                   "Final Test ROUGE-L Score": final_rL_score, "Final Test BERT Score": final_b_score,
                   "Final Test BLEU Score": final_bl_score})
    end_time = time.time(); total_time = end_time - start_time
    tqdm.write(f"Total execution time: {total_time:.2f} seconds")
    final_meta_file.write(f"Total execution time: {total_time:.2f} seconds\n")
    if 'wandb' in globals() and wandb.run:
        wandb.log({"Total Execution Time (seconds)": total_time})
        wandb.save(os.path.join(args.meta_dir, args.meta_name))
        if args.write_preds:
            pname = args.meta_name.split('.')[0] + "_predictions.json"
            ppath = os.path.join(args.meta_dir, pname)
            if os.path.exists(ppath): wandb.save(ppath)

global_progress_bar.close()
if 'wandb' in globals() and wandb.run:
    wandb.finish()
if 'meta_file_initial_open' in globals() and meta_file_initial_open and not meta_file_initial_open.closed:
    meta_file_initial_open.close()

del model
del tokenizer
torch.cuda.empty_cache()
gc.collect()
tqdm.write("Bloomz model and tokenizer unloaded. Script finished.")